1) Import everything we need

In [109]:

# Modules for model workflows and transformer building
import numpy as np
import random
import torch
from torch import nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer, TransformerDecoderLayer, TransformerDecoder
from torch.utils.data import DataLoader
from torch.utils.data.dataset import TensorDataset
from torch.optim import Adam
from math import floor

# Modules for data generation
from scipy.signal import cont2discrete

# Modules for some custom loss function
from torch.linalg import inv

2) Setup seed for reproducibility

In [110]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

3) Define transformer class

In [111]:
class TransformerAutoencoder(nn.Module):
    def __init__(self, 
                 encoder_input_dim, 
                 decoder_input_dim, 
                 hidden_dim,
                 num_heads, 
                 encoder_embedding_dim, 
                 decoder_embedding_dim,
                 num_layers, 
                 dropout):
        super(TransformerAutoencoder, self).__init__()
        self.encoder_input_dim = encoder_input_dim
        self.decoder_input_dim = decoder_input_dim
        self.encoder_embedding_dim = encoder_embedding_dim
        self.decoder_embedding_dim = decoder_embedding_dim
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.dropout = dropout

        # Encoder Embedding
        self.encoder_embedding = nn.Linear(self.encoder_input_dim, 
                                           self.encoder_embedding_dim)

        # Encoder
        self.encoder_layer = TransformerEncoderLayer(d_model=self.encoder_embedding_dim,
                                                     nhead=self.num_heads,
                                                     dim_feedforward=self.hidden_dim,
                                                     dropout=self.dropout,
                                                     batch_first=True)
        self.encoder = TransformerEncoder(self.encoder_layer,
                                          num_layers=self.num_layers)

        # Decoder Embedding
        self.decoder_embedding = nn.Linear(self.decoder_input_dim, 
                                           self.decoder_embedding_dim)

        # Decoder
        self.decoder_layer = TransformerDecoderLayer(d_model=self.decoder_embedding_dim,
                                                     nhead=self.num_heads,
                                                     dim_feedforward=self.hidden_dim,
                                                     dropout=self.dropout,
                                                     batch_first=True)
        self.decoder = TransformerDecoder(self.decoder_layer, 
                                          num_layers=self.num_layers)

        # Final output layer
        self.out = nn.Linear(self.decoder_embedding_dim,
                             self.decoder_input_dim)

    def forward(self, inputs, targets):        
        # Encode the input
        encoded_input = self.encoder(self.encoder_embedding(inputs))
        
        # Decode the target
        decoder_input = self.decoder_embedding(targets)
        decoded_target = self.decoder(decoder_input, encoded_input)        
        
        # Apply the final output layer
        target_output = self.out(decoded_target)
        return target_output

4) Data for our experiments: Lorrentz data, Linear springs data, Kuramoto Shivasinsky

a) Lorrentz Data: 
n = num of sequence
allInputs = Y_data = measurements [nx1]
allTargets = X_data = states [nx3]

In [112]:
rawData = np.load("../Data/lorenz_data_hackathon.npz")
allInputs = torch.Tensor(rawData["Y_data"])
allTargets = torch.Tensor(rawData["X_data"])
numSequence = allInputs.shape[0]
encoder_input_dim = allInputs.shape[2]
decoder_input_dim = allTargets.shape[2]

b) Linear springs data:
n = num of sequence
allInputs = Y_data = measurements [nx1]
allTargets = X_data = states [nx3]

In [113]:
rawData = np.load("../Data/linear_springs_data.npz")
allInputs = torch.Tensor(rawData["Y_data"])
allTargets = torch.Tensor(rawData["X_data"])
A = torch.Tensor(rawData["A_matrix"])
H = torch.Tensor(rawData["H_matrix"])
Q = torch.Tensor(rawData["Q_matrix"])
R = torch.Tensor(rawData["R_matrix"])
P = torch.Tensor(rawData["P_matrix"])
def f(x): return A @ x
def h(x): return H @ x
numSequence = allInputs.shape[0]
encoder_input_dim = allInputs.shape[2]
decoder_input_dim = allTargets.shape[2]

5) Split the data into training, validation and testing

In [114]:
trainPercent = 0.8
testPercent = 0.1
validatePercent = 0.1
lenTrainingData = floor(trainPercent*numSequence)
lenTestingData = floor(testPercent*numSequence)
lenValidatingData = numSequence-lenTestingData-lenTrainingData

print("## Split the data:\n")
print(f"Training data set size   : {lenTrainingData}\n",
      f"Validation data set size : {lenTestingData}\n",
      f"Test data set size       : {lenValidatingData}")

trainingTensorDataSet = TensorDataset(allInputs[0:lenTestingData-1],
                                      allTargets[0:lenTestingData-1])
testingTensorDataSet = TensorDataset(allInputs[lenTestingData:lenTestingData+lenTrainingData-1],
                                     allTargets[lenTestingData:lenTestingData+lenTrainingData-1])
validatingTensorDataSet = TensorDataset(allInputs[lenTestingData+lenTestingData:],
                                        allTargets[lenTestingData+lenTestingData:])

## Split the data:

Training data set size   : 400
 Validation data set size : 50
 Test data set size       : 50


6) Define MSE loss function evaluator. Do back propagation as we evaluate the loss for every dataset we train

In [115]:
def evaluateMSELoss(model, optimizer, train_dataloader):
    sum_loss = 0
    criterion = nn.MSELoss()
    for i, batch in enumerate(train_dataloader):
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs, targets)
        loss = criterion(outputs, targets)
        sum_loss+=loss.item() 
        loss.backward()
        optimizer.step()
    avg_loss = sum_loss/len(train_dataloader)
    return avg_loss

def evaluateCustomLoss1(model, optimizer, train_dataloader, alpha):
    sum_loss = 0
    criterion = nn.MSELoss()
    for i, batch in enumerate(train_dataloader):
        inputs, targets = batch
        optimizer.zero_grad()
        outputs = model(inputs, targets)
        h_of_outputs = 0.0*outputs
        for j in range(outputs.shape[1]):
            h_of_outputs[0,j,:] = h(outputs[0,j,:]) 
        loss = alpha * criterion(outputs, targets) + (1.0 - alpha)*criterion(inputs, h_of_outputs)
        sum_loss+=loss.item() 
        loss.backward()
        optimizer.step()
    avg_loss = sum_loss/len(train_dataloader)
    return avg_loss

7) Define network training and validating parameters

In [116]:
alpha = 0.81
num_layers = 8
num_epochs = 10
hidden_dim = 8
num_heads = 4
encoder_embedding_dim = 16
decoder_embedding_dim = 16
learn_rate = 0.01
weight_decay = 0.005
dropout = 0.05

trainDataSetLoader =  DataLoader(trainingTensorDataSet, batch_size = 1)
validationDataSetLoader = DataLoader(validatingTensorDataSet)
testDataSetLoader = DataLoader(testingTensorDataSet)

# Data dict to store best results
best_result = dict({
    'model':[],
    'learn_rate':[],
    'num_epochs':[],
    'encoder_embedding_dim':[],
    'decoder_embedding_dim':[],
    'hidden_dim':[],
    'num_heads':[],
    'weight_decay':[],
    'dropout':[],
    'avg_training_loss':[],
    'best_validation_loss':[]
})

8) Define model and optimizer

In [117]:
model = TransformerAutoencoder(encoder_input_dim, 
                               decoder_input_dim,
                               hidden_dim, 
                               num_heads, 
                               encoder_embedding_dim,
                               decoder_embedding_dim, 
                               num_layers, 
                               dropout)
optimizer = Adam(model.parameters(), 
                 lr=learn_rate, 
                 weight_decay=weight_decay)
    
# initialise the best and avg losses
best_validation_loss = 1000000.
avg_training_loss = 0.
avg_validation_loss = 0.

for epoch in range(num_epochs):
    print(f"EPOCH NUMBER: {epoch+1}")
    model.train(True)
    avg_training_loss = evaluateMSELoss(model, optimizer, trainDataSetLoader)
    model.eval()
    
    sum_validation_loss = 0.0

    with torch.no_grad():
        for i, batch in enumerate(validationDataSetLoader):
            inputs, targets = batch
            outputs = model(inputs, targets)
            criterion = nn.MSELoss()
            val_loss = criterion(outputs, targets)
            sum_validation_loss += val_loss.item()
    avg_validation_loss = sum_validation_loss / len(validationDataSetLoader)
    
    print(f"AVERAGE TRAINING LOSS  : {avg_training_loss}\nAVERAGE VALIDATION LOSS: {avg_validation_loss}")

    if avg_validation_loss < best_validation_loss:
        best_validation_loss = avg_validation_loss
        print(f"BEST VALIDATION LOSS: {best_validation_loss} at EPOCH {epoch+1}")
        best_result['model']=model.state_dict()
        best_result['learn_rate']=learn_rate
        best_result['encoder_embedding_dim']=encoder_embedding_dim
        best_result['decoder_embedding_dim'] =decoder_embedding_dim
        best_result['num_epochs']=num_epochs
        best_result['hidden_dim']=hidden_dim
        best_result['num_heads']=num_heads
        best_result['weight_decay']=weight_decay
        best_result['dropout']=dropout
        best_result['avg_training_loss']=avg_training_loss
        best_result['best_validation_loss']=best_validation_loss

EPOCH NUMBER: 1
AVERAGE TRAINING LOSS  : 0.05319876033736735
AVERAGE VALIDATION LOSS: 0.02401241793297231
BEST VALIDATION LOSS: 0.02401241793297231 at EPOCH 1
EPOCH NUMBER: 2
AVERAGE TRAINING LOSS  : 0.017646162846714865
AVERAGE VALIDATION LOSS: 0.01124260839424096
BEST VALIDATION LOSS: 0.01124260839424096 at EPOCH 2
EPOCH NUMBER: 3
AVERAGE TRAINING LOSS  : 0.010819737780459074
AVERAGE VALIDATION LOSS: 0.005916205552057363
BEST VALIDATION LOSS: 0.005916205552057363 at EPOCH 3
EPOCH NUMBER: 4
AVERAGE TRAINING LOSS  : 0.008256644276635987
AVERAGE VALIDATION LOSS: 0.00666369928221684
EPOCH NUMBER: 5
AVERAGE TRAINING LOSS  : 0.010744945488261933
AVERAGE VALIDATION LOSS: 0.008772781880106778
EPOCH NUMBER: 6
AVERAGE TRAINING LOSS  : 0.010923780016220954
AVERAGE VALIDATION LOSS: 0.01288377859047614
EPOCH NUMBER: 7
AVERAGE TRAINING LOSS  : 0.011151769806687929
AVERAGE VALIDATION LOSS: 0.011040630453499034
EPOCH NUMBER: 8
AVERAGE TRAINING LOSS  : 0.01598751043178597
AVERAGE VALIDATION LOSS: 0.0

9) Test the unseen data

In [125]:
model.eval()
total_loss = 0.0
total_seqs = 0

criterion = nn.MSELoss()

for batch in testDataSetLoader:
    y_seq, x_true = batch  
    b, T, m = x_true.shape
    x0 = torch.zeros_like(x_true[:, 0, :])  
    # model.InitSequence(x0, T)
    x_pred = model(y_seq[:, 1:],x_true[:,1:])   
    x_gt = x_true[:, 1:, :]           
    loss = criterion(x_pred, x_gt)  
    total_loss += loss.item()

final_mse = total_loss 
print(f"[MSE] = {final_mse:.6f}")

torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])
torch.Size([1, 100, 20])


In [119]:
print(x_true.shape)

torch.Size([1, 100, 20])
